### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp04'
DATA_URLS = {'serres_assets.gpkg': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp04/serres_assets.gpkg'}
SCHEMA = {'serres_assets.gpkg': {'kind': 'gpkg', 'layers': ['parks', 'water_points']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP04 — Explore GeoPackage

Ανοίγω → Εξερευνώ → Κατανοώ → Φιλτράρω → Επαληθεύω → Εξάγω

Το GeoPackage είναι container, όχι απαραίτητα ένα layer. Ανακάλυψε πρώτα τα layer names και έπειτα έλεγξε κάθε layer χωριστά: schema, attributes, CRS, geometry, bounds και map.

In [ ]:
import geopandas as gpd
from pathlib import Path
gpkg_path = DATA_ROOT / "gp04/serres_assets.gpkg"
assert gpkg_path.exists()


## 1. Layers

Πρώτα βλέπουμε τι περιέχει το GeoPackage.

In [ ]:
# layers = gpd.list_layers(gpkg_path)
# print(layers)


## 2. Ανάγνωση layer και schema

In [ ]:
# parks = gpd.read_file(gpkg_path, layer="parks")
# water = gpd.read_file(gpkg_path, layer="water_points")
# Εμφάνισε rows, columns, dtypes και missing values


## 3. CRS, geometry types και bounds

In [ ]:
# Έλεγξε CRS, geom_type.value_counts(), total_bounds και is_valid
# Κάνε plot τα δύο layers στον ίδιο άξονα


## 4. Filtering και QA

In [ ]:
# active_water = water[water["status"] == "active"].copy()
# Πρόσθεσε layer-specific checks και κάνε export σε GeoJSON


## Mini Challenge

Κράτησε active water points, επιβεβαίωσε ότι είναι Points και εξήγαγε το αποτέλεσμα.

In [ ]:
# Γράψε εδώ τη λύση σου
